In [1]:
%pip install -U audio-separator pydub onnxruntime
%pip uninstall -y samplerate

  Using cached samplerate-0.1.0-py2.py3-none-any.whl.metadata (3.2 kB)
Using cached samplerate-0.1.0-py2.py3-none-any.whl (4.0 MB)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Found existing installation: samplerate 0.1.0
Uninstalling samplerate-0.1.0:
  Successfully uninstalled samplerate-0.1.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
from pathlib import Path
from audio_separator.separator import Separator
from pydub import AudioSegment

SUPPORTED_FORMATS = {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}

def validate_input(input_path: str) -> Path:
    path = Path(input_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {input_path}")
    if path.suffix.lower() not in SUPPORTED_FORMATS:
        raise ValueError(f"Unsupported format '{path.suffix}'. Supported: {SUPPORTED_FORMATS}")
    return path

def identify_stems(output_files: list, output_dir: str) -> tuple[str, str]:
    vocals_path, crowd_path = None, None
    for f in output_files:
        full = os.path.join(output_dir, f)
        lower = f.lower()
        if "vocal" in lower:
            vocals_path = full
        elif "instrumental" in lower or "no_vocal" in lower:
            crowd_path = full
    if not vocals_path or not crowd_path:
        print("  [Warning] Could not auto-detect stems by name. Falling back to index order.")
        crowd_path  = os.path.join(output_dir, output_files[0])
        vocals_path = os.path.join(output_dir, output_files[1])
    return vocals_path, crowd_path

def print_audio_info(label: str, segment: AudioSegment):
    duration = len(segment) / 1000
    print(f"  {label}: {duration:.1f}s | {segment.channels}ch | {segment.frame_rate}Hz")

def process_sports_audio(
    input_path: str,
    preference: str = "mute_commentary",
    crowd_boost_db: float = 10.0,
    commentary_reduce_db: float = 15.0,
    output_dir: str = "./output",
    model_name: str = "UVR-MDX-NET-Voc_FT.onnx",
):
    input_file = validate_input(input_path)
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n{'='*50}")
    print(f"  Input : {input_file.name}")
    print(f"  Mode  : {preference}")
    print(f"  Model : {model_name}")
    print(f"{'='*50}\n")

    print("[1/3] Running source separation...")
    separator = Separator(output_dir=output_dir, output_format="WAV")
    separator.load_model(model_filename=model_name)
    output_files = separator.separate(str(input_file))
    print(f"  Stems produced: {output_files}")

    print("\n[2/3] Loading stems...")
    commentary_path, crowd_path = identify_stems(output_files, output_dir)
    crowd = AudioSegment.from_wav(crowd_path)
    commentary = AudioSegment.from_wav(commentary_path)
    print_audio_info("Crowd stem      ", crowd)
    print_audio_info("Commentary stem ", commentary)

    print(f"\n[3/3] Applying preference: '{preference}'...")
    if preference == "mute_commentary":
        final_output = crowd
    elif preference == "amplify_crowd":
        final_output = (crowd + crowd_boost_db).overlay(commentary - commentary_reduce_db)
    elif preference == "balanced":
        final_output = (crowd + 5).overlay(commentary)
    else:
        raise ValueError(f"Unknown preference '{preference}'")

    result_name = f"sports_{preference}_{input_file.stem}.wav"
    final_output.export(result_name, format="wav")
    print(f"\n✅ Done! Saved as '{result_name}'")
    return result_name

if __name__ == "__main__":
    pass

In [5]:
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory

VIDEO_FORMATS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}

def process_sports_video(
    video_path: str,
    preference: str = "mute_commentary",
    crowd_boost_db: float = 10.0,
    commentary_reduce_db: float = 15.0,
    output_dir: str = "./output",
    model_name: str = "UVR-MDX-NET-Voc_FT.onnx",
) -> str:
    video_file = Path(video_path)
    if not video_file.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")
    if video_file.suffix.lower() not in VIDEO_FORMATS:
        raise ValueError(f"Unsupported video format '{video_file.suffix}'. Supported: {VIDEO_FORMATS}")

    output_dir_path = Path(output_dir)
    output_dir_path.mkdir(parents=True, exist_ok=True)

    result_name = output_dir_path / f"sports_{preference}_{video_file.stem}.mp4"

    with TemporaryDirectory() as tmp:
        tmp_dir = Path(tmp)
        extracted_audio = tmp_dir / f"{video_file.stem}_audio.wav"

        # 1) Extract video audio as WAV for stem separation
        extract_cmd = [
            "ffmpeg", "-y",
            "-i", str(video_file),
            "-vn",
            "-ac", "2",
            "-ar", "44100",
            str(extracted_audio),
]
        subprocess.run(extract_cmd, check=True, capture_output=True, text=True)

        # 2) Apply existing sports audio separation/remix pipeline
        remixed_audio = process_sports_audio(
            input_path=str(extracted_audio),
            preference=preference,
            crowd_boost_db=crowd_boost_db,
            commentary_reduce_db=commentary_reduce_db,
            output_dir=output_dir,
            model_name=model_name,
)

        # 3) Replace original audio track with remixed output
        mux_cmd = [
            "ffmpeg", "-y",
            "-i", str(video_file),
            "-i", str(remixed_audio),
            "-c:v", "copy",
            "-c:a", "aac",
            "-map", "0:v:0",
            "-map", "1:a:0",
            "-shortest",
            str(result_name),
]
        subprocess.run(mux_cmd, check=True, capture_output=True, text=True)

    print(f"\n✅ Video done! Saved as '{result_name}'")
    return str(result_name)

# Example:
# process_sports_video(video_path="match.mp4", preference="mute_commentary")

In [3]:
process_sports_audio(input_path='input/commentary.mp3')

2026-04-08 20:28:09,463 - INFO - separator - Separator version 0.44.1 instantiating with output_dir: ./output, output_format: WAV
2026-04-08 20:28:09,463 - INFO - separator - Using model directory from model_file_dir parameter: /tmp/audio-separator-models/
2026-04-08 20:28:09,464 - INFO - separator - Operating System: Darwin Darwin Kernel Version 24.6.0: Mon Jul 14 11:30:40 PDT 2025; root:xnu-11417.140.69~1/RELEASE_ARM64_T6041
2026-04-08 20:28:09,465 - INFO - separator - System: Darwin Node: Dhanas-MacBook-Pro.local Release: 24.6.0 Machine: arm64 Proc: arm
2026-04-08 20:28:09,465 - INFO - separator - Python Version: 3.14.0
2026-04-08 20:28:09,465 - INFO - separator - PyTorch Version: 2.11.0
2026-04-08 20:28:09,570 - INFO - separator - FFmpeg installed: ffmpeg version 8.1 Copyright (c) 2000-2026 the FFmpeg developers
2026-04-08 20:28:09,572 - INFO - separator - ONNX Runtime CPU package installed with version: 1.24.4
2026-04-08 20:28:09,597 - INFO - separator - Apple Silicon MPS/CoreML i


  Input : commentary.mp3
  Mode  : mute_commentary
  Model : UVR-MDX-NET-Voc_FT.onnx

[1/3] Running source separation...


2026-04-08 20:28:11,591 - INFO - separator - Load model duration: 00:00:01
2026-04-08 20:28:11,591 - INFO - separator - Processing file: input/commentary.mp3
2026-04-08 20:28:11,592 - INFO - separator - Starting separation process for audio_file_path: input/commentary.mp3
2026-04-08 20:28:11,595 - INFO - common_separator - Input audio subtype: MPEG_LAYER_III
2026-04-08 20:28:11,595 - WARNING - common_separator - Unknown audio subtype MPEG_LAYER_III, defaulting to 16-bit output
2026-04-08 20:28:11,596 - INFO - common_separator - Detected input bit depth: 16-bit
100%|██████████| 12/12 [00:00<00:00, 16.82it/s]
2026-04-08 20:28:23,375 - INFO - mdx_separator - Saving Instrumental stem to commentary_(Instrumental)_UVR-MDX-NET-Voc_FT.wav...
2026-04-08 20:28:23,401 - INFO - common_separator - Audio duration is 0.02 hours (64.45 seconds).
2026-04-08 20:28:23,401 - INFO - common_separator - Using pydub for writing.
2026-04-08 20:28:23,406 - INFO - common_separator - Writing output with 16-bit de

  Stems produced: ['commentary_(Instrumental)_UVR-MDX-NET-Voc_FT.wav', 'commentary_(Vocals)_UVR-MDX-NET-Voc_FT.wav']

[2/3] Loading stems...
  Crowd stem      : 64.5s | 2ch | 44100Hz
  Commentary stem : 64.5s | 2ch | 44100Hz

[3/3] Applying preference: 'mute_commentary'...

✅ Done! Saved as 'sports_mute_commentary_commentary.wav'


'sports_mute_commentary_commentary.wav'

In [8]:
process_sports_video("input/match_clip.mp4", preference="amplify_crowd")

2026-04-08 20:31:33,479 - INFO - separator - Separator version 0.44.1 instantiating with output_dir: ./output, output_format: WAV
2026-04-08 20:31:33,480 - INFO - separator - Using model directory from model_file_dir parameter: /tmp/audio-separator-models/
2026-04-08 20:31:33,480 - INFO - separator - Operating System: Darwin Darwin Kernel Version 24.6.0: Mon Jul 14 11:30:40 PDT 2025; root:xnu-11417.140.69~1/RELEASE_ARM64_T6041
2026-04-08 20:31:33,480 - INFO - separator - System: Darwin Node: Dhanas-MacBook-Pro.local Release: 24.6.0 Machine: arm64 Proc: arm
2026-04-08 20:31:33,481 - INFO - separator - Python Version: 3.14.0
2026-04-08 20:31:33,481 - INFO - separator - PyTorch Version: 2.11.0
2026-04-08 20:31:33,505 - INFO - separator - FFmpeg installed: ffmpeg version 8.1 Copyright (c) 2000-2026 the FFmpeg developers
2026-04-08 20:31:33,507 - INFO - separator - ONNX Runtime CPU package installed with version: 1.24.4
2026-04-08 20:31:33,507 - INFO - separator - Apple Silicon MPS/CoreML i


  Input : match_clip_audio.wav
  Mode  : amplify_crowd
  Model : UVR-MDX-NET-Voc_FT.onnx

[1/3] Running source separation...


2026-04-08 20:31:34,621 - INFO - separator - Load model duration: 00:00:01
2026-04-08 20:31:34,621 - INFO - separator - Processing file: /var/folders/v6/frwp0yjs06j_wm3461r0j6rr0000gp/T/tmpfl565xyn/match_clip_audio.wav
2026-04-08 20:31:34,621 - INFO - separator - Starting separation process for audio_file_path: /var/folders/v6/frwp0yjs06j_wm3461r0j6rr0000gp/T/tmpfl565xyn/match_clip_audio.wav
2026-04-08 20:31:34,622 - INFO - common_separator - Input audio subtype: PCM_16
2026-04-08 20:31:34,622 - INFO - common_separator - Detected input bit depth: 16-bit
100%|██████████| 39/39 [00:02<00:00, 16.44it/s]
2026-04-08 20:32:05,117 - INFO - mdx_separator - Saving Instrumental stem to match_clip_audio_(Instrumental)_UVR-MDX-NET-Voc_FT.wav...
2026-04-08 20:32:05,145 - INFO - common_separator - Audio duration is 0.06 hours (218.43 seconds).
2026-04-08 20:32:05,145 - INFO - common_separator - Using pydub for writing.
2026-04-08 20:32:05,160 - INFO - common_separator - Writing output with 16-bit de

  Stems produced: ['match_clip_audio_(Instrumental)_UVR-MDX-NET-Voc_FT.wav', 'match_clip_audio_(Vocals)_UVR-MDX-NET-Voc_FT.wav']

[2/3] Loading stems...
  Crowd stem      : 218.4s | 2ch | 44100Hz
  Commentary stem : 218.4s | 2ch | 44100Hz

[3/3] Applying preference: 'amplify_crowd'...

✅ Done! Saved as 'sports_amplify_crowd_match_clip_audio.wav'

✅ Video done! Saved as 'output/sports_amplify_crowd_match_clip.mp4'


'output/sports_amplify_crowd_match_clip.mp4'